# Question 4


In [0]:
from pyspark.sql.functions import *

df = spark.read.csv(
    '/Volumes/cyntexa_dev/sales/my_volume/*.csv',
    inferSchema=True,
    header=True
).withColumns({
    'file_name': col('_metadata.file_name'),
    'file_path': col('_metadata.file_path'),
    'ingestion_time': current_timestamp(),
    'ingestion_date': current_date()
})
df.write.format('delta').mode('overwrite').saveAsTable('cyntexa_dev.sales.sales_table_py')

In [0]:
%sql

select * from cyntexa_dev.sales.sales_table_py;

# Question 5

```root
 |-- customer: struct (nullable = true)
 |    |-- address: struct (nullable = true)
 |    |    |-- city: string (nullable = true)
 |    |    |-- country: string (nullable = true)
 |    |    |-- postal_code: string (nullable = true)
 |    |    |-- state: string (nullable = true)
 |    |    |-- street: string (nullable = true)
 |    |-- customer_id: string (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- membership: string (nullable = true)
 |    |-- name: string (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- attributes: struct (nullable = true)
 |    |    |    |-- capacity: string (nullable = true)
 |    |    |    |-- color: string (nullable = true)
 |    |    |    |-- size: string (nullable = true)
 |    |    |    |-- warranty_months: long (nullable = true)
 |    |    |    |-- wattage: long (nullable = true)
 |    |    |-- category: string (nullable = true)
 |    |    |-- item_id: string (nullable = true)
 |    |    |-- product_name: string (nullable = true)
 |    |    |-- quantity: long (nullable = true)
 |    |    |-- sku: string (nullable = true)
 |    |    |-- unit_price: double (nullable = true)
 |-- payment: struct (nullable = true)
 |    |-- amount_paid: double (nullable = true)
 |    |-- currency: string (nullable = true)
 |    |-- method: string (nullable = true)
 |    |-- provider: string (nullable = true)
 |    |-- status: string (nullable = true)
 |-- store: struct (nullable = true)
 |    |-- location_name: string (nullable = true)
 |    |-- region: string (nullable = true)
 |    |-- store_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- transaction_id: string (nullable = true)
 ```
+

In [0]:
from pyspark.sql.functions import * 

df = spark.read.option("multiLine", True).json(
    '/Volumes/cyntexa_dev/sales/my_volume/*.json'
)

df.printSchema()


# Explode the array column into individual struct rows
df_exploded = df.select(
    col("transaction_id"),
    col("timestamp"),
    col("customer"),
    col("store"),
    col("payment"),
    explode(col("items")).alias("item")
)
df_exploded.display()

In [0]:
df_flat = df_exploded.select(
    # Top-level fields
    col("transaction_id"),
    col("timestamp"),
    
    # Nested Struct: Customer
    col("customer.customer_id").alias("customer_id"),
    col("customer.name").alias("customer_name"),
    col("customer.email").alias("customer_email"),

    # col("customer.address.street").alias("customer_street"),
    # col("customer.address.city").alias("customer_city"),

    col("customer.address.country").alias("customer_country"),
    
    # Nested Struct: Store
    col("store.store_id").alias("store_id"),
    col("store.region").alias("store_region"),
    col("store.location_name").alias("store_location"),
    
    # Exploded & Nested Struct: Item
    col("item.item_id").alias("item_id"),
    col("item.sku").alias("item_sku"),
    col("item.attributes.warranty_months").alias("warranty_months"),
    
    # Nested Struct: Payment
    col("payment.amount_paid").alias("amount_paid"),
    col("payment.currency").alias("currency")
)

df_flat.display()

# Question 6

In [0]:
from pyspark.sql import functions as F

# 1. Create a dummy DataFrame with enough data
data = [(i, f"user_{i}", i % 5, i * 100) for i in range(1000)]
df = spark.createDataFrame(data, ["id", "name", "group_id", "score"])

# 2. Build a multi-step transformation chain
df_chain = (
    df
    .filter(F.col("score") > 200)              
    .withColumn("double_score", F.col("score") * 2)
    .groupBy("group_id")                        
    .agg(F.avg("double_score").alias("avg_score"))
    .filter(F.col("avg_score") == 100100)           
)

# 3. Print the execution plan
df_chain.explain(True)